# Visualize Best Agent

This notebook loads a population checkpoint (`latest.pkl`), extracts the best agent from it, displays its program code, and visualizes it playing FlappyBird with human rendering.


In [1]:
# Import required libraries
import flappy_bird_env  # noqa
import numpy as np
import os
import pickle
from pathlib import Path
from scipy.special import expit

# Disable headless mode for human rendering
os.environ.pop('SDL_VIDEODRIVER', None)

try:
    import pygame
    pygame.init()
    print("✓ Pygame initialized for human rendering")
except Exception as e:
    print(f"Warning: Could not initialize pygame: {e}")

import gymnasium as gym
from memory_system import MemoryConfig, MemoryType
from evaluator import FlappyBirdEvaluator, FlappyBirdEvaluatorConfig


✓ Pygame initialized for human rendering


## Load Best Agent from Population Checkpoint

Load the checkpoint file (`latest.pkl`) which contains a population, then extract the best agent from it.


In [2]:
# Path to checkpoint file (contains a population)
checkpoint_path = "gen_0847.pkl"

# Import EvolutionEngine to load checkpoint
from evolution_engine import EvolutionEngine

# Check if file exists
if not Path(checkpoint_path).exists():
    print(f"❌ Checkpoint file not found: {checkpoint_path}")
    print("Please update checkpoint_path to point to your checkpoint pickle file.")
    best_agent = None
    generation = None
    population = None
else:
    print(f"Loading checkpoint from: {checkpoint_path}")
    
    # Load the checkpoint (contains population and metadata)
    checkpoint = EvolutionEngine.load_checkpoint(checkpoint_path)
    population = checkpoint['population']
    generation = checkpoint.get('generation', 'unknown')
    checkpoint_fitness = checkpoint.get('fitness', None)
    
    print(f"✓ Checkpoint loaded successfully!")
    print(f"  Generation: {generation}")
    if checkpoint_fitness is not None:
        print(f"  Best fitness in checkpoint: {checkpoint_fitness:.4f}")
    print(f"  Population size: {len(population)}")
    
    # Get the best agent from the population
    # Prefer best_ever if available (best agent across all generations)
    if population.best_ever is not None:
        best_agent = population.best_ever
        best_generation = population.best_ever_generation
        print(f"  Using best_ever agent (found at generation {best_generation})")
    else:
        best_agent = population.get_best()
        print(f"  Using best agent from current generation")
    
    print(f"\n✓ Best agent extracted!")
    print(f"  Agent ID: {best_agent.id}")
    if best_agent.fitness is not None:
        print(f"  Fitness: {best_agent.fitness:.4f}")
    else:
        print(f"  Fitness: not set")
    print(f"  Program length: {len(best_agent.program)}")
    print(f"  Age: {best_agent.age}")
    if best_agent.parent_ids:
        print(f"  Parent IDs: {best_agent.parent_ids}")


Loading checkpoint from: gen_0847.pkl
✓ Checkpoint loaded successfully!
  Generation: 847
  Best fitness in checkpoint: 1.1480
  Population size: 500
  Using best_ever agent (found at generation 847)

✓ Best agent extracted!
  Agent ID: 338954
  Fitness: 1.1480
  Program length: 147
  Age: 0
  Parent IDs: (319549, 338537)


## Display Program Code

Show the full program and the effective program (with introns removed).


In [3]:
# Reconstruct MemoryConfig from the agent's MemoryBank
# This is needed for instruction string formatting - use agent's ACTUAL dimensions!
memory = best_agent.memory
memory_cfg = MemoryConfig(
    n_scalar=memory.n_scalar,
    n_vector=memory.n_vector,
    n_matrix=memory.n_matrix,
    n_obs_scalar=memory.n_obs_scalar,
    n_obs_vector=memory.n_obs_vector,
    n_obs_matrix=memory.n_obs_matrix,
    vector_size=memory.vector_size,
    matrix_shape=memory.matrix_shape,
)

# Output registers (default for FlappyBird, adjust if your config uses different)
output_registers = [(MemoryType.SCALAR, 0)]

print("="*80)
print("FULL PROGRAM")
print("="*80)
print()

for i, instr in enumerate(best_agent.program.instructions):
    print(f"{i:4d}: {instr.to_resolved_str(memory_cfg)}")

print()
print("="*80)
print(f"Total: {len(best_agent.program)} instructions")
print("="*80)


FULL PROGRAM

   0: scalar[7→7] = automl_scalar_div(scalar[3504→0], obs_matrix[3→3][6974→(17,2)])
   1: vector[1→1] = automl_vector_add(vector[6611→3], obs_matrix[1→1][:,8982→15])
   2: scalar[2→2] = automl_scalar_sin(scalar[1130→2])
   3: matrix[3→3] = cv_gaussian_blur(obs_matrix[1→1], scalar[1822→6], scalar[4860→4])
   4: matrix[5→5] = automl_scalar_matrix_mul(scalar[7850→2], matrix[9749→5])
   5: matrix[7→7] = cv_sobel_y(matrix[7921→1], scalar[705→1])
   6: scalar[4→4] = automl_scalar_min(scalar[5473→1], scalar[5643→3])
   7: matrix[5→5] = cv_avg_pool(matrix[8045→5], scalar[8818→2])
   8: scalar[0→0] = automl_scalar_cos(scalar[5537→1])
   9: scalar[4→4] = automl_vector_norm(obs_matrix[1→1][:,7092→15])
  10: scalar[6→6] = automl_matrix_mean(matrix[2329→1])
  11: scalar[4→4] = automl_vector_dot(obs_matrix[3→3][:,3020→17], obs_matrix[1→1][:,9027→18])
  12: scalar[4→4] = automl_scalar_arctan(scalar[3496→0])
  13: scalar[4→4] = automl_scalar_sub(scalar[7030→6], scalar[9556→4])
  14: scal

In [4]:
# Show effective program (with introns removed)
effective_program = best_agent.get_effective_program(output_registers)

print("="*80)
print("EFFECTIVE PROGRAM (INTRONS REMOVED)")
print("="*80)
print()

if len(effective_program.instructions) > 0:
    for i, instr in enumerate(effective_program.instructions):
        print(f"{i:4d}: {instr.to_resolved_str(memory_cfg)}")
else:
    print("  (No effective instructions found)")

print()
print("="*80)
print(f"Effective: {len(effective_program)} instructions (out of {len(best_agent.program)} total)")
if len(best_agent.program) > 0:
    effective_ratio = len(effective_program) / len(best_agent.program)
    print(f"Effective code rate: {effective_ratio:.3f} ({effective_ratio*100:.1f}%)")
print("="*80)


EFFECTIVE PROGRAM (INTRONS REMOVED)

   0: scalar[0→0] = automl_scalar_sub(scalar[7928→0], scalar[6734→6])

Effective: 1 instructions (out of 147 total)
Effective code rate: 0.007 (0.7%)


## Create Evaluator with Human Rendering

Set up the FlappyBird evaluator with human rendering mode to visualize the agent playing.


In [5]:
# Check agent's memory dimensions to determine training strategy
print(f"Agent's vector size: {best_agent.memory.vector_size}")
print(f"Agent's matrix shape: {best_agent.memory.matrix_shape}")

# Determine strategy based on memory dimensions
# Trinary strategy produces 21x21 matrices, feature_vector produces vectors
if best_agent.memory.matrix_shape == (21, 21) and best_agent.memory.vector_size == 21:
    strategy = "trinary"
    print("✓ Detected: Agent was trained with 'trinary' strategy")
else:
    strategy = "feature_vector"
    print(f"✓ Detected: Agent was trained with '{strategy}' strategy")

# Create evaluator config for visualization
# Use the same config as training, but with human rendering
evaluator_config = FlappyBirdEvaluatorConfig(
    env_id="FlappyBird-v0",
    episodes=3,  # Run 3 episodes to see the agent play
    max_steps=500,
    output_register=0,
    render_mode="human",  # Human rendering to see the game
    rng_seed=0,
    patch_strategy=strategy,  # Match the training strategy
    color_channel=2,  # Not used for trinary, but required
    normalize=False,  # Trinary uses raw values (-1, 0, 1)
    quantization_factor=0.03,  # Not used for trinary
    feature_vector_size=64,  # Not used for trinary
    frame_stack_size=4,  # CRITICAL: Must match training! Agent expects 4 stacked frames
    # Trinary strategy parameters (match training config)
    trinary_crop_bottom=100,
    trinary_resize_factor=0.03,
    trinary_final_size=21,
    trinary_bird_h_min=0,
    trinary_bird_h_max=50,
    trinary_bird_s_min=50,
    trinary_bird_s_max=255,
    trinary_bird_v_min=50,
    trinary_bird_v_max=255,
    trinary_pipe_h_min=35,
    trinary_pipe_h_max=45,
    trinary_pipe_s_min=40,
    trinary_pipe_s_max=255,
    trinary_pipe_v_min=40,
    trinary_pipe_v_max=255,
    output_registers=[(MemoryType.SCALAR, 0)],
    n_jobs=1,  # Sequential for visualization
)

print("\nCreating FlappyBird evaluator with human rendering...")
evaluator = FlappyBirdEvaluator(config=evaluator_config)
print("✓ Evaluator created!")
print(f"  Episodes: {evaluator.episodes}")
print(f"  Max steps per episode: {evaluator.max_steps}")
print(f"  Render mode: {evaluator.config.render_mode}")
print(f"  Patch strategy: {strategy}")
print()
print("⚠️  NOTE: FlappyBird windows will appear when you run the next cell!")
print("   Close the windows or press Ctrl+C to stop.")


Agent's vector size: 21
Agent's matrix shape: (21, 21)
✓ Detected: Agent was trained with 'trinary' strategy

Creating FlappyBird evaluator with human rendering...
✓ Evaluator created!
  Episodes: 3
  Max steps per episode: 500
  Render mode: human
  Patch strategy: trinary

⚠️  NOTE: FlappyBird windows will appear when you run the next cell!
   Close the windows or press Ctrl+C to stop.


## Visualize Agent Playing

Run the agent and watch it play FlappyBird. The game windows will appear showing the agent's performance.


In [6]:
# Run the agent and visualize
print("="*80)
print("RUNNING BEST AGENT")
print("="*80)
print()

total_reward = 0.0
episode_rewards = []

for episode_idx in range(evaluator.episodes):
    print(f"\nEpisode {episode_idx + 1}/{evaluator.episodes}")
    print("-" * 80)
    
    # Seed the episode
    episode_seed = int((evaluator.config.rng_seed + episode_idx * 100) % (2**31))
    observation, _ = evaluator.env.reset(seed=episode_seed)
    observation = np.asarray(observation, dtype=np.float32)
    
    # Reset evaluator's frame buffer for this episode (important for trinary/quantized strategies)
    evaluator.frame_buffer = None
    
    # Copy memory for this episode (preserves evolved constants in working registers)
    # Observations will be loaded fresh each step, so no need to reset observation registers
    memory = best_agent.memory.copy()
    episode_reward = 0.0
    steps = 0
    
    for step in range(evaluator.max_steps):
        # Process observation
        # Note: _process_observation returns different formats based on strategy:
        # - feature_vector: returns dict {'vector': [...], 'matrix': [...]}
        # - quantized/full_image: returns tuple (observations_list, 'matrix')
        result = evaluator._process_observation(observation)
        
        # Load observations into memory
        if isinstance(result, dict):
            # feature_vector strategy returns both vector and matrix
            memory.load_observation(result)
        else:
            # quantized/full_image strategy returns tuple
            processed_observations, obs_type = result
            if obs_type == 'vector':
                memory.load_observation({'vector': processed_observations})
            else:
                memory.load_observation({'matrix': processed_observations})
        
        # Execute the program
        best_agent.program.execute(memory,debug=True)
        
        # Read action from output register
        action_value = memory.read_scalar(evaluator.output_register)
        normalized = expit(action_value)  # Sigmoid
        action = 1 if normalized >= 0.5 else 0
        
        # Take step in environment
        observation, reward, terminated, truncated, _ = evaluator.env.step(action)
        observation = np.asarray(observation, dtype=np.float32)
        episode_reward += reward
        steps += 1
        
        if terminated or truncated:
            break
    
    episode_rewards.append(episode_reward)
    total_reward += episode_reward
    
    print(f"  Steps: {steps}")
    print(f"  Reward: {episode_reward:.2f}")
    print(f"  Action value (scalar[0]): {action_value:.4f}")
    print(f"  Normalized (sigmoid): {normalized:.4f}")
    print(f"  Action chosen: {'FLAP' if action == 1 else 'NOOP'}")

print()
print("="*80)
print("SUMMARY")
print("="*80)
print(f"Total episodes: {evaluator.episodes}")
print(f"Average reward: {total_reward / evaluator.episodes:.2f}")
print(f"Rewards per episode: {[f'{r:.2f}' for r in episode_rewards]}")
print("="*80)

# Close the evaluator
evaluator.close()
print("\n✓ Visualization complete!")


RUNNING BEST AGENT


Episode 1/3
--------------------------------------------------------------------------------
Executing instruction 0: scalar[7] = automl_scalar_div(scalar[3504], obs_matrix[3]→scalar[6974])
Executing instruction 1: vector[1] = automl_vector_add(vector[6611], obs_matrix[1]→vector[8982])
Executing instruction 2: scalar[2] = automl_scalar_sin(scalar[1130])
Executing instruction 3: matrix[3] = cv_gaussian_blur(obs_matrix[1]→matrix[3887], scalar[1822], scalar[4860])
Executing instruction 4: matrix[5] = automl_scalar_matrix_mul(scalar[7850], matrix[9749])
Executing instruction 5: matrix[7] = cv_sobel_y(matrix[7921], scalar[705])
Executing instruction 6: scalar[4] = automl_scalar_min(scalar[5473], scalar[5643])
Executing instruction 7: matrix[5] = cv_avg_pool(matrix[8045], scalar[8818])
Executing instruction 8: scalar[0] = automl_scalar_cos(scalar[5537])
Executing instruction 9: scalar[4] = automl_vector_norm(obs_matrix[1]→vector[7092])
Executing instruction 10: scalar[6]

## Additional Information

Display additional details about the best agent's memory and constants.


In [7]:
# Display memory information
print("="*80)
print("BEST AGENT MEMORY INFORMATION")
print("="*80)
print()

memory = best_agent.memory

print("Scalar registers (working):")
for i in range(min(8, memory.n_scalar)):
    print(f"  scalar[{i}]: {memory.scalars[i]:.6f}")

print()
print("Vector registers (working):")
for i in range(min(3, memory.n_vector)):
    vec_str = ", ".join([f"{v:.3f}" for v in memory.vectors[i][:5]])
    if len(memory.vectors[i]) > 5:
        vec_str += "..."
    print(f"  vector[{i}]: [{vec_str}]")

print()
print("Matrix registers (working):")
for i in range(min(3, memory.n_matrix)):
    print(f"  matrix[{i}]: shape {memory.matrices[i].shape}, "
          f"mean={memory.matrices[i].mean():.4f}, "
          f"std={memory.matrices[i].std():.4f}")

print()
print("Observation registers:")
if memory.n_obs_matrix > 0:
    print(f"  obs_matrix[0]: shape {memory.obs_matrices[0].shape}")

print("="*80)


BEST AGENT MEMORY INFORMATION

Scalar registers (working):
  scalar[0]: -0.738389
  scalar[1]: -0.545987
  scalar[2]: -0.982616
  scalar[3]: -0.337583
  scalar[4]: -2.063522
  scalar[5]: 1.162288
  scalar[6]: 0.821936
  scalar[7]: 1.898992

Vector registers (working):
  vector[0]: [-0.261, 0.112, -1.536, 0.678, 0.527...]
  vector[1]: [0.397, 1.541, 1.128, -0.012, -0.386...]
  vector[2]: [-0.500, 0.123, -0.007, 0.252, 0.651...]

Matrix registers (working):
  matrix[0]: shape (21, 21), mean=-0.0057, std=0.3896
  matrix[1]: shape (21, 21), mean=0.0192, std=0.3848
  matrix[2]: shape (21, 21), mean=-0.0128, std=0.3747

Observation registers:
  obs_matrix[0]: shape (21, 21)


## Action Distribution Analysis

Analyze the action distribution of the best agent - how often it chooses to flap vs. not flap.


In [8]:
# Analyze action distribution for the best agent
print("="*80)
print("ACTION DISTRIBUTION ANALYSIS")
print("="*80)
print()

# Create a new evaluator config for analysis (no human rendering, faster)
analysis_config = FlappyBirdEvaluatorConfig(
    env_id="FlappyBird-v0",
    episodes=10,  # Run more episodes for better statistics
    max_steps=500,
    output_register=0,
    render_mode="rgb_array",  # No human rendering for speed
    rng_seed=42,
    patch_strategy=strategy,  # Use detected strategy
    color_channel=2,
    normalize=False,
    quantization_factor=0.03,
    feature_vector_size=64,
    frame_stack_size=4,  # CRITICAL: Must match training!
    # Trinary strategy parameters (match training config)
    trinary_crop_bottom=100,
    trinary_resize_factor=0.03,
    trinary_final_size=21,
    trinary_bird_h_min=0,
    trinary_bird_h_max=50,
    trinary_bird_s_min=50,
    trinary_bird_s_max=255,
    trinary_bird_v_min=50,
    trinary_bird_v_max=255,
    trinary_pipe_h_min=35,
    trinary_pipe_h_max=45,
    trinary_pipe_s_min=40,
    trinary_pipe_s_max=255,
    trinary_pipe_v_min=40,
    trinary_pipe_v_max=255,
    output_registers=[(MemoryType.SCALAR, 0)],
    n_jobs=1,
)

analysis_evaluator = FlappyBirdEvaluator(config=analysis_config)

# Track actions across all episodes
all_actions = []
all_action_values = []
all_normalized_values = []
episode_rewards = []
episode_actions_list = []  # Store actions per episode for detailed analysis

print("Running agent for action distribution analysis...")
print()

for episode_idx in range(analysis_evaluator.episodes):
    # Seed the episode
    episode_seed = int((analysis_evaluator.config.rng_seed + episode_idx * 100) % (2**31))
    observation, _ = analysis_evaluator.env.reset(seed=episode_seed)
    observation = np.asarray(observation, dtype=np.float32)
    
    # Reset frame buffer
    analysis_evaluator.frame_buffer = None
    
    # Copy memory for this episode
    memory = best_agent.memory.copy()
    episode_reward = 0.0
    episode_actions = []
    episode_action_values = []
    episode_normalized = []
    
    for step in range(analysis_evaluator.max_steps):
        # Process observation
        result = analysis_evaluator._process_observation(observation)
        
        # Load observations into memory
        if isinstance(result, dict):
            memory.load_observation(result)
        else:
            processed_observations, obs_type = result
            if obs_type == 'vector':
                memory.load_observation({'vector': processed_observations})
            else:
                memory.load_observation({'matrix': processed_observations})
        
        # Execute the program (no debug output)
        best_agent.program.execute(memory)
        
        # Read action from output register
        action_value = memory.read_scalar(analysis_evaluator.output_register)
        normalized = expit(action_value)  # Sigmoid
        action = 1 if normalized >= 0.5 else 0
        
        # Track actions
        episode_actions.append(action)
        episode_action_values.append(action_value)
        episode_normalized.append(normalized)
        all_actions.append(action)
        all_action_values.append(action_value)
        all_normalized_values.append(normalized)
        
        # Take step in environment
        observation, reward, terminated, truncated, _ = analysis_evaluator.env.step(action)
        observation = np.asarray(observation, dtype=np.float32)
        episode_reward += reward
        
        if terminated or truncated:
            break
    
    episode_rewards.append(episode_reward)
    episode_actions_list.append(episode_actions)  # Store for per-episode analysis
    flap_rate = 100*sum(episode_actions)/len(episode_actions) if len(episode_actions) > 0 else 0
    print(f"Episode {episode_idx + 1}: {len(episode_actions)} steps, "
          f"Reward: {episode_reward:.2f}, "
          f"Actions: {sum(episode_actions)} flaps / {len(episode_actions)} total "
          f"({flap_rate:.1f}% flap rate)")

analysis_evaluator.close()

print()
print("="*80)
print("ACTION DISTRIBUTION SUMMARY")
print("="*80)
print()

total_steps = len(all_actions)
flap_count = sum(all_actions)
noop_count = total_steps - flap_count
flap_percentage = 100 * flap_count / total_steps if total_steps > 0 else 0
noop_percentage = 100 * noop_count / total_steps if total_steps > 0 else 0

print(f"Total steps analyzed: {total_steps}")
print(f"Episodes: {analysis_evaluator.episodes}")
print()
print("Action Distribution:")
print(f"  FLAP (action=1):  {flap_count:6d} steps ({flap_percentage:5.2f}%)")
print(f"  NOOP (action=0):  {noop_count:6d} steps ({noop_percentage:5.2f}%)")
print()

# Action value statistics
action_values_array = np.array(all_action_values)
normalized_array = np.array(all_normalized_values)

print("Action Value Statistics (from scalar[0]):")
print(f"  Mean:   {action_values_array.mean():.4f}")
print(f"  Std:    {action_values_array.std():.4f}")
print(f"  Min:    {action_values_array.min():.4f}")
print(f"  Max:    {action_values_array.max():.4f}")
print(f"  Median: {np.median(action_values_array):.4f}")
print()

print("Normalized Value Statistics (after sigmoid):")
print(f"  Mean:   {normalized_array.mean():.4f}")
print(f"  Std:    {normalized_array.std():.4f}")
print(f"  Min:    {normalized_array.min():.4f}")
print(f"  Max:    {normalized_array.max():.4f}")
print(f"  Median: {np.median(normalized_array):.4f}")
print()

# Reward statistics
rewards_array = np.array(episode_rewards)
print("Reward Statistics:")
print(f"  Mean reward per episode:   {rewards_array.mean():.4f}")
print(f"  Std reward per episode:    {rewards_array.std():.4f}")
print(f"  Min reward per episode:    {rewards_array.min():.4f}")
print(f"  Max reward per episode:    {rewards_array.max():.4f}")
print(f"  Total reward:              {rewards_array.sum():.4f}")
print()

# Action pattern analysis
print("Action Pattern Analysis:")
consecutive_flaps = 0
max_consecutive_flaps = 0
consecutive_noops = 0
max_consecutive_noops = 0

for i, action in enumerate(all_actions):
    if action == 1:
        consecutive_flaps += 1
        consecutive_noops = 0
        max_consecutive_flaps = max(max_consecutive_flaps, consecutive_flaps)
    else:
        consecutive_noops += 1
        consecutive_flaps = 0
        max_consecutive_noops = max(max_consecutive_noops, consecutive_noops)

print(f"  Max consecutive flaps:  {max_consecutive_flaps}")
print(f"  Max consecutive noops:  {max_consecutive_noops}")
print()

# Flap rate per episode
print("Flap Rate Per Episode:")
for i, episode_actions in enumerate(episode_actions_list):
    episode_flaps = sum(episode_actions)
    episode_total = len(episode_actions)
    episode_flap_rate = 100 * episode_flaps / episode_total if episode_total > 0 else 0
    print(f"  Episode {i+1}: {episode_flaps}/{episode_total} flaps ({episode_flap_rate:.1f}%) - Reward: {episode_rewards[i]:.2f}")

# Simple visualization
print("="*80)
print("VISUALIZATION")
print("="*80)
print()

# Create a simple bar chart using text
bar_length = 50
flap_bars = int(flap_percentage / 100 * bar_length)
noop_bars = bar_length - flap_bars

print("Action Distribution (visual):")
print(f"  FLAP: {'█' * flap_bars} {flap_percentage:.1f}%")
print(f"  NOOP: {'█' * noop_bars} {noop_percentage:.1f}%")
print()

# Threshold analysis
threshold = 0.5
above_threshold = np.sum(normalized_array >= threshold)
below_threshold = np.sum(normalized_array < threshold)

print(f"Normalized values >= 0.5 (would flap):  {above_threshold} ({100*above_threshold/len(normalized_array):.2f}%)")
print(f"Normalized values < 0.5 (would noop):   {below_threshold} ({100*below_threshold/len(normalized_array):.2f}%)")
print()

print("="*80)


ACTION DISTRIBUTION ANALYSIS

Running agent for action distribution analysis...

Episode 1: 83 steps, Reward: 0.08, Actions: 14 flaps / 83 total (16.9% flap rate)
Episode 2: 83 steps, Reward: 0.08, Actions: 14 flaps / 83 total (16.9% flap rate)
Episode 3: 83 steps, Reward: 0.08, Actions: 7 flaps / 83 total (8.4% flap rate)
Episode 4: 182 steps, Reward: 1.18, Actions: 15 flaps / 182 total (8.2% flap rate)
Episode 5: 91 steps, Reward: 0.09, Actions: 17 flaps / 91 total (18.7% flap rate)
Episode 6: 83 steps, Reward: 0.08, Actions: 8 flaps / 83 total (9.6% flap rate)
Episode 7: 83 steps, Reward: 0.08, Actions: 14 flaps / 83 total (16.9% flap rate)
Episode 8: 99 steps, Reward: 1.10, Actions: 16 flaps / 99 total (16.2% flap rate)
Episode 9: 83 steps, Reward: 0.08, Actions: 14 flaps / 83 total (16.9% flap rate)
Episode 10: 83 steps, Reward: 0.08, Actions: 8 flaps / 83 total (9.6% flap rate)

ACTION DISTRIBUTION SUMMARY

Total steps analyzed: 953
Episodes: 10

Action Distribution:
  FLAP (acti